# (Colored) Pixel fonts rabbit hole

I had an idea of using the Jetpack's font (defined in `JETPACK0.DAT`) as a proper font on a webpage. This is not trivial, and the rabbit hole is quite deep.

## SVG Fonts

SVG fonts are dead. Only WebKit has ever implemented it. Chrome/Blink removed support for it. Firefox decided not to implement it.

* <https://caniuse.com/svg-fonts>
* <https://chromestatus.com/feature/5930075908210688>
* <https://bugzilla.mozilla.org/show_bug.cgi?id=119490>

## Colors in OpenType

SVG Fonts are dead. Long live SVG In OpenType. Long live COLR/CPAL tables. Wait, what?

Since SVG Fonts are going nowhere, there are two competing alternatives. (I'm already ignoring one or two other solutions that aren't gaining any traction.)

* SVG in OpenType
* COLR/CPAL tables in OpenType
    * v0 is widely implemented
    * v1 addes gradients and other extra features
* CBDT/CBLC adds bitmaps to fonts, implemented by Google in Android
* SBIX adds raster images to fonts, implemented by Apple in Mac and iOS

There is/was also a proposal for embedding bitmaps into fonts, but this isn't gaining enough support.

Browser and tooling support is a mess. A lot of articles online are outdated.

* <https://caniuse.com/colr> claims COLR v0 widely supported.
* <https://caniuse.com/colr-v1> claims COLR v1 not supported on Apple WebKit.
* <https://www.colorfonts.wtf/> claims COLR format is the only one widely supported.
* <https://pixelambacht.nl/chromacheck> tests each format.
* <https://www.high-logic.com/font-editor/fontcreator/tutorials/create-opentype-color-fonts> shows a comparison of formats across several browsers and applications.

From my own tests

| Browser | COLR v0 | COLR v1 | OpenType-SVG | SBIX | CBDT/CBLC |
| - | - | - | - | - | - |
| Firefox 152 on Linux/Mac | COLR v0 | COLR v1 | OpenType-SVG | - | - |
| Chrome  150 on Linux/Mac | COLR v0 | COLR v1 | - | SBIX | CBDT/CBLC |
| Edge    150 on Linux/Mac | COLR v0 | COLR v1 | - | SBIX | CBDT/CBLC |
| Safari 26.5 on Mac | COLR v0 | - | OpenType-SVG | SBIX | - |

### SVG and COLR tooling

[BirdFont](https://birdfont.org/download.php) has multiple licenses. The "Plus" license adds support for SVG/COLR/CPAL. It's quite affordable, and with a reasonable pricing model.

[FontCreator](https://www.high-logic.com/font-editor/fontcreator/tutorials/create-opentype-color-fonts) is an expensive (but with lifetime license) Windows/Mac application that supports COLRv0 (but not v1). It has limited support for SVG (can add glyphs, but cannot edit them).

[Glyphs](https://glyphsapp.com/buy) is a quite expensive Mac application that supports both [CPAL/COLR](https://glyphsapp.com/learn/creating-a-microsoft-color-font) and [SVG](https://glyphsapp.com/learn/creating-an-svg-color-font).

[FontStruct](https://fontstruct.com/about) is a gratis online font-building tool. There isn't enough documentation about its capabilities, but [it support colors](https://fontstruct.com/news/2021/06/17/layers-and-color/) for [patrons](https://fontstruct.com/faq/88/fs-patrons/all-about-fs-patrons). It is very easy to use. Really easy. So easy that I quickly made a font: <https://fontstruct.com/fontstructions/show/2919379>

### SVG in OpenType tooling

[Xerographer Fonts](https://xerographer.github.io/) are examples of embedding SVGs into OpenType, [including source-code](https://github.com/xerographer/multicoloure-font), which uses [scfbuild SVGinOT Color Font Builder](https://github.com/13rac1/scfbuild) tool to build a font out of dozens of individual SVG files. [Chromium/Blink do not support these](https://issues.chromium.org/issues/40336440).

There was a Windows-only [OpenType-SVG-Font-Editor](https://github.com/microsoft/OpenType-SVG-Font-Editor) written in C# by interns at Microsoft. It is [still available at the Microsoft Store](https://apps.microsoft.com/detail/9nj7k9jx60p1).

[Fontself Maker](https://www.fontself.com/make-fonts-on-desktop) is an extension to Adobe Illustrator and Adobe Photoshop to create fonts. It seems to support OpenType-SVG fonts.

### COLR tooling

So far, I found no tooling that only supports COLR but not SVG.

### Tooling without any extra features

[FontForge](https://fontforge.org/) is a FLOSS GUI tool to design fonts. [It does not support COLR](https://github.com/fontforge/fontforge/issues/677), and likely never will. It is unknown about SVG in OpenType.

[RetroNick's Vector Font Editor](https://retronick2020.itch.io/vectorfonteditor) is a Windows tool to create and design old-style `*.FON` files. It supports no fancy features. The code was written with help of AI.

### See also

* <https://pixelambacht.nl/2014/multicolor-fonts/> - over ten years old, the comparison is outdated
* <https://robert.ocallahan.org/2013/02/svg-in-opentype-new-approach-to-svg.html>
* <https://learn.microsoft.com/en-us/typography/opentype/spec/svg>
* <https://learn.microsoft.com/en-us/typography/opentype/spec/colr>
* <https://learn.microsoft.com/en-us/typography/opentype/spec/cpal>
* <https://freetype.org/freetype2/docs/reference/ft2-svg_fonts.html>

## CSS features

It is possible to define multiple font palettes in CSS, and switch them dynamically. That allows different colors (or color variations) on the same font file.

* [font-palette and @font-palette-values](https://github.com/drott/csswg-drafts/blob/paletteExplainer/css-fonts-4/palette-explainer.md)
* <https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/Properties/font-palette>
* <https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@font-palette-values>
* <https://developer.mozilla.org/en-US/docs/Web/CSS/Reference/At-rules/@font-palette-values/override-colors>
* <https://github.com/o-t-w/color-font-palette/blob/main/styles.css>

## Cool idea(s)

It would be nice to have a tool that reads `JETPACK0.DAT` and converts that font into a web-compatible font format. Even better if it preserves the original colors.

Embedding raster image sounds like a simple solution, but I'm not sure about overall compatibility (not supported in Firefox), about how good it looks (probably blurry unless the pixel size is perfectly alignes), or about being future-proof.

Converting those characters into vector is feasible, but much more difficult and time-consuming. But it should be possible to write code that generates SVG "pixels", and then use `scfbuild` to combine those SVGs into a font. Unfortunately, this is not supported in Blink, which means the majority of the browsers.

The best (?) solution is to probably convert them to COLR with an associated CPAL. Unfortunately, as of "right now", I'm not aware of any tooling I could use to achieve that. And I really don't want to write my own tool for that. (That's a pretty deep rabbit hole.)